# Answer the vignettes (gold tier)

A clinician reads short patient texts and answers the questions about them. Run the two cells below,
then use the buttons: **no commands to remember**.

1. Fill in your **expert id** (for example `E01`) and role, and click **Start / resume**.
   Vignettes come in a random order that is fixed for your id.
2. **Cite the passages that matter.** Copy words from the patient text, paste them under it and click
   **Add citation**. Each citation gets a number and a colour, and needs a **direction**: toward
   ADHD-related traits, away from them, neutral, or unclear. Press **Show** on a citation to highlight
   its passage in the text, and **Hide** to clear it. Nothing else is highlighted.
3. **Answer one question at a time, citing as you write.** **Cite** adds the citation's number to the
   end of your answer, and you can also type a number yourself, like `[2]`, anywhere in it. Each answer
   needs a minimum of different words; citation numbers don't count.
4. **Q3 ends with Lock in results.** It asks once more, then **Q1–Q3 and your citations so far are
   locked**: they record your reading before you saw what comes next. You can still add new citations
   after this.
5. **Q4** names one more fact about the writer (a *what-if*). Write your working hypothesis in one
   sentence, then pick how it changes, from **−2** (*my hypothesis is strongly reinforced*) through
   **0** (*unchanged*) to **+2** (*changes completely*). For +1 or +2 also say which way it moves, and
   for anything but 0 say why. At 0, the hypothesis and the scale are enough: click **Next**.
6. Q5 is a free answer, Q6 a rating from 1 to 5 (or *insufficient information*).
7. **Review** lets you edit Q4–Q6, then **Accept & next vignette**.
8. **Skip vignette** puts it aside (its button turns orange; click it to come back). Click any vignette
   button to jump to that vignette.
9. Your draft is saved on every move, so you can stop at any time. Run the cells again with the same
   expert id and click **Start / resume** to carry on.
10. When you are done, **Finish & write document** (asks once more) writes the session to
    `data/fixtures_chips/synthetic/gold/documents/`. Skipped or unfinished vignettes are left out.

This follows the web platform's flow (`notes/frontend-vignettes.md`), with buttons in place of selecting
and hovering. The rules are in `scripts/elicitation/chip_answers.py`. The terminal app,
`scripts/elicitation/answer_vignettes.py`, still asks the older way, without citations, so the two can't
resume each other's drafts.

In [ ]:
import html
import json
import sys
import time
from pathlib import Path

import ipywidgets as W
from IPython.display import display

# The kernel starts in notebooks/; find the repository root from here.
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "elicitation").is_dir()
)
sys.path[:0] = [str(ROOT / "backend"), str(ROOT / "scripts" / "elicitation")]

import answer_vignettes as av  # noqa: E402  (drafts and the manifest, shared with the terminal app)
import chip_answers as ca  # noqa: E402  (citations, the lock-in and the document)

from app.vignettes.questions import (  # noqa: E402
    Q4_HYPOTHESIS,
    Q4_TOWARD,
    Q4_WHY,
    QUESTIONS,
    SHIFT_LABELS,
    SHIFT_LEAD,
    TOWARD_LABELS,
)

print("repository:", ROOT)

In [ ]:
INKS = ["#ffd66b", "#ffb8a3", "#a6e8c5", "#d5c4ff", "#9fe3e8", "#ffbbd9", "#d9ec8c", "#e8cfa6"]
DIRECTIONS = [
    ("Direction…", None),
    ("Toward traits", "toward"),
    ("Away from traits", "away"),
    ("Neutral", "neutral"),
    ("Unclear", "unclear"),
]

CSS = """<style>
.av-box{border:1px solid var(--jp-border-color2,#bbb);border-radius:6px;padding:10px 14px;
  background:var(--jp-layout-color2,#f5f5f5);color:var(--jp-ui-font-color1,inherit);
  white-space:pre-wrap;line-height:1.5;max-height:340px;overflow:auto}
.av-box mark{color:#16233b;padding:0 1px;border-radius:2px}
.av-chip{display:inline-block;min-width:1.4em;text-align:center;color:#16233b;font-weight:600;
  font-size:.85em;line-height:1.35;padding:0 3px;border-left:2px solid rgba(22,35,59,.7);
  border-radius:0 3px 3px 0}
.av-q{font-weight:600;margin:6px 0 2px}
.av-muted{opacity:.7;font-size:.9em}
.av-quote{white-space:normal;line-height:1.35}
.av-warn{color:#c77700}.av-ok{color:#2e7d32}.av-bad{color:#c62828}
.av-whatif{font-weight:600;border-left:3px solid var(--jp-brand-color1,#1976d2);padding-left:8px}
</style>"""


def ink(chip: int) -> str:
    return INKS[(chip - 1) % len(INKS)]


def chip_html(chip: int) -> str:
    return f'<span class="av-chip" style="background:{ink(chip)}">{chip}</span>'


def answer_html(stored: str) -> str:
    """A saved answer as HTML, with its chips shown as coloured numbers."""
    out, pos = [], 0
    for match in ca.CHIP_RE.finditer(stored):
        out += [html.escape(stored[pos : match.start()]), chip_html(int(match.group(1)))]
        pos = match.end()
    return "".join([*out, html.escape(stored[pos:])])


def highlight(text: str, citation: dict | None) -> str:
    """`text` as HTML, with only the citation being shown marked."""
    if citation is None:
        return html.escape(text)
    start, end = citation["span"]
    return (
        f"{html.escape(text[:start])}"
        f'<mark style="background:{ink(citation["chip"])}">{html.escape(text[start:end])}</mark>'
        f"{html.escape(text[end:])}"
    )


def button(label, icon="", style="", **kw):
    return W.Button(description=label, icon=icon, button_style=style, **kw)


class AnswerDashboard:
    """Buttons instead of commands: start, cite, move between questions and vignettes, lock in,
    review, finish. The draft is saved on every move and (every few seconds) while typing."""

    SAVE_EVERY = 3.0  # seconds between disk writes while typing

    def __init__(self):
        tiers = [(f"{ca.SOURCE}/{tier}", (ca.SOURCE, tier)) for tier in ca.source_tiers()]
        style = {"description_width": "110px"}
        self.expert = W.Text(description="Expert id", placeholder="E01", style=style)
        self.role = W.Text(
            description="Role",
            placeholder="consultant psychiatrist, 12 years adult ADHD",
            style=style,
            layout=W.Layout(width="560px"),
        )
        self.source = W.Dropdown(
            description="Vignettes from",
            options=tiers,
            value=(ca.SOURCE, "silver") if (ca.SOURCE, "silver") in [t[1] for t in tiers] else None,
            style=style,
        )
        self.order = W.Dropdown(description="Order", options=["shuffle", "fixed"], style=style)
        self.start_btn = button("Start / resume", "play", "primary")
        self.start_msg = W.HTML()
        self.start_box = W.VBox(
            [
                W.HTML("<h3>Answer the vignettes</h3>"),
                self.expert,
                self.role,
                W.HBox([self.source, self.order]),
                self.start_btn,
                self.start_msg,
            ]
        )

        self.title = W.HTML()
        self.progress = W.IntProgress(min=0, max=1, layout=W.Layout(width="260px"))
        self.counts = W.HTML()
        self.jump = W.HBox(layout=W.Layout(flex_flow="row wrap"))
        self.vignette = W.HTML()
        wide = W.Layout(width="100%")
        self.quote = W.Text(
            placeholder="Paste words copied from the text above",
            layout=W.Layout(flex="1 1 auto", width="auto"),
        )
        self.add_btn = button("Add citation", "plus", layout=W.Layout(width="130px", flex="0 0 auto"))
        self.quote_row = W.HBox([self.quote, self.add_btn], layout=W.Layout(gap="8px"))
        self.citations = W.VBox()
        self.qhead = W.HTML()
        self.text = W.Textarea(rows=10, layout=wide, placeholder="Your answer")
        self.helper = W.HTML()
        self.rating = W.ToggleButtons(
            options=[*[(str(n), n) for n in range(1, 6)], ("Insufficient information", "i")],
            value=None,
        )
        # Q4: a -2..+2 scale (ToggleButtons: a slider cannot be left unset), hypothesis, direction
        # (only for +1/+2) and why (only when not 0)
        self.q4_hyp = W.Text(
            placeholder="e.g. sleep loss after the move, not clearly more", layout=wide
        )
        self.q4_shift = W.ToggleButtons(
            options=[(f"{k:+d} {lbl}" if k else f"0 {lbl}", k) for k, lbl in SHIFT_LABELS.items()],
            value=None,
            style={"button_width": "auto"},
        )
        self.q4_toward = W.ToggleButtons(
            options=[(lbl, k) for k, lbl in TOWARD_LABELS.items()],
            value=None,
            style={"button_width": "auto"},
        )
        self.q4_other = W.Text(placeholder="Which explanation?", layout=wide)
        self.q4_why = W.Textarea(rows=6, layout=wide, placeholder="Why does it change?")
        self.q4_helper = W.HTML()
        self.q4_box = W.VBox()
        self.review = W.VBox()
        self.area = W.VBox()
        self.back_btn = button("Back", "arrow-left")
        self.next_btn = button("Next", "arrow-right", "primary")
        self.skip_btn = button("Skip vignette", "forward", "warning")
        self.save_btn = button("Save draft", "save")
        self.finish_btn = button("Finish & write document", "check-circle", "success")
        self.msg = W.HTML()
        self.main = W.VBox(
            [
                W.HBox([self.title, self.progress, self.counts], layout=W.Layout(gap="16px")),
                self.jump,
                self.vignette,
                self.quote_row,
                self.citations,
                self.qhead,
                self.area,
                W.HBox(
                    [self.back_btn, self.next_btn, self.skip_btn, self.save_btn, self.finish_btn],
                    layout=W.Layout(gap="8px", margin="10px 0"),
                ),
                self.msg,
            ],
            layout=W.Layout(display="none"),
        )
        self.root = W.VBox([W.HTML(CSS), self.start_box, self.main])

        self.expert.observe(self._prefill_role, "value")
        self.start_btn.on_click(lambda _: self._start())
        self.add_btn.on_click(lambda _: self._add_citation())
        self.text.observe(self._on_text, "value")
        self.rating.observe(self._on_rating, "value")
        for name, widget in [
            ("hypothesis", self.q4_hyp),
            ("shift", self.q4_shift),
            ("toward", self.q4_toward),
            ("other_explanation", self.q4_other),
        ]:
            widget.observe(lambda c, name=name: self._on_q4(name, c["new"]), "value")
        self.q4_why.observe(self._on_q4_why, "value")
        self.back_btn.on_click(lambda _: self._back())
        self.next_btn.on_click(lambda _: self._next())
        self.skip_btn.on_click(lambda _: self._skip())
        self.save_btn.on_click(lambda _: self._pause())
        self.finish_btn.on_click(lambda _: self._finish())
        self.vid: str | None = None
        self.steps: list[str] = []
        self.k = 0
        self.edit_from_review = False
        self.confirm = False
        self.confirm_lock = False
        self.confirm_remove = None
        self.shown_chip = None
        self.loading = False
        self.last_save = 0.0

    # --- state ---------------------------------------------------------------------------------

    @property
    def entry(self) -> dict:
        return self.draft["answers"][self.vid]

    def entry_for(self, vid: str) -> dict:
        return self.draft["answers"][vid]

    @property
    def cited(self) -> list[dict]:
        return self.entry.setdefault("highlights", [])

    def _locked_count(self) -> int:
        return self.entry.get("locked_highlights", 0) if av.gate_locked(self.entry) else 0

    def _done(self) -> list[str]:
        return [v for v in self.draft["order"] if self.draft["answers"].get(v, {}).get("done")]

    def _pending(self) -> list[str]:
        gone = {*self._done(), *self.draft["skipped"]}
        return [v for v in self.draft["order"] if v not in gone]

    def _save(self) -> None:
        av.write_json(av.draft_path_for(self.gold, self.draft["expert_id"]), self.draft)
        self.last_save = time.monotonic()

    def _say(self, text: str, kind: str = "muted") -> None:
        self.msg.value = f'<div class="av-{kind}">{text}</div>'

    def _cancel_confirmations(self) -> None:
        self.confirm = False
        self.confirm_lock = False
        self.confirm_remove = None

    # --- start ---------------------------------------------------------------------------------

    def _prefill_role(self, change) -> None:
        if not self.source.value:
            return
        path = av.fixture_dir("gold", ca.SOURCE, av.FIXTURES) / "experts.json"
        if path.exists() and not self.role.value:
            self.role.value = json.loads(path.read_text()).get(change["new"].strip(), "")

    def _start(self) -> None:
        if not self.source.value:
            self.start_msg.value = (
                '<span class="av-bad">No vignettes found in data/fixtures_chips/synthetic.</span>'
            )
            return
        try:
            self.gold, self.vignettes, self.draft = ca.open_session(
                self.expert.value.strip(),
                self.role.value.strip() or None,
                self.source.value[1],
                self.order.value,
            )
        except (av.SetupError, ValueError) as e:
            self.start_msg.value = f'<span class="av-bad">{html.escape(str(e))}</span>'
            return
        self.source_name = self.source.value[1]
        self.start_box.layout.display = "none"
        self.main.layout.display = ""
        self.progress.max = len(self.draft["order"])
        self.title.value = f"<b>{html.escape(self.draft['expert_id'])}</b>"
        self._save()
        pending = self._pending()
        self._open(pending[0] if pending else self.draft["order"][0])

    # --- opening a vignette and moving around --------------------------------------------------

    def _open(self, vid: str) -> None:
        self.vid, v = vid, self.vignettes[vid]
        if vid in self.draft["skipped"]:
            self.draft["skipped"].remove(vid)
        entry = self.draft["answers"].setdefault(vid, {"started_at": av.now_iso(), "answers": {}})
        self.steps = [*v.question_ids, ca.REVIEW]
        # A finished vignette opens on its review.
        self.k = self.steps.index(ca.REVIEW if entry.get("done") else ca.first_open_step(v, entry))
        self.edit_from_review = False
        self.shown_chip = None
        self._cancel_confirmations()
        self._save()
        self._render()

    def current_vignette(self):
        """The vignette on screen; every caller runs after one has been opened."""
        assert self.vid is not None, "no vignette is open"
        return self.vignettes[self.vid]

    def _go(self, k: int) -> None:
        v, target = self.current_vignette(), self.steps[k]
        if target not in av.gate_ids(v) and not av.gate_locked(self.entry):
            return self._say("Lock in your results first: the next questions build on them.", "warn")
        self.k = k
        self.shown_chip = None
        self._cancel_confirmations()
        self._save()
        self._render()

    def _back(self) -> None:
        self.edit_from_review = False
        if self.k > 0:
            self._go(self.k - 1)

    def _next(self) -> None:
        v, step = self.current_vignette(), self.steps[self.k]
        if step == ca.REVIEW:
            return self._accept()
        locked = av.gate_locked(self.entry) and step in av.gate_ids(v)
        if not locked and (problem := ca.step_problem(v, self.entry, step)):
            return self._say(html.escape(problem), "warn")
        if self._is_lock_step():
            return self._lock_in()
        self._say("")
        last = len(self.steps) - 1
        goto = last if self.edit_from_review else self.k + 1
        self.edit_from_review = False
        self._go(goto)

    def _is_lock_step(self) -> bool:
        gate = av.gate_ids(self.current_vignette())
        return bool(gate) and self.steps[self.k] == gate[-1] and not av.gate_locked(self.entry)

    def _lock_in(self) -> None:
        v = self.current_vignette()
        if problem := ca.lock_problem(v, self.entry):
            return self._say(html.escape(problem), "warn")
        if not self.confirm_lock:
            self.confirm_lock = True
            self.next_btn.description = "Confirm: lock in results"
            self.next_btn.button_style = "danger"
            count = len(self.cited)
            kept = "your citation" if count == 1 else f"your {count} citations" if count else ""
            lines = [
                f"You won't be able to change your answers to Q1 to Q3{' or ' + kept if kept else ''}"
                + (", including direction." if kept else ".")
                + " The remaining questions open next, and you can still add new citations there.",
                *ca.lock_warnings(self.entry),
                "Click again to confirm, or anything else to cancel.",
            ]
            return self._say("<br>".join(html.escape(line) for line in lines), "warn")
        ca.lock_in(self.entry)
        self._say("Results locked in.", "ok")
        self._go(self.k + 1)

    def _accept(self) -> None:
        v, e = self.current_vignette(), self.entry
        if problem := ca.accept_problem(v, e):
            return self._say(html.escape(problem), "warn")
        e["done"], e["submitted_at"] = True, av.now_iso()
        self._save()
        pending = self._pending()
        if pending:
            self._say(f"{self.vid} saved.", "ok")
            self._open(pending[0])
        else:
            self._say("All vignettes are handled. Click “Finish & write document”.", "ok")
            self._render()

    def _skip(self) -> None:
        if self.vid not in self.draft["skipped"]:
            self.draft["skipped"].append(self.vid)
        self.entry.pop("done", None)
        self._save()
        pending = self._pending()
        if pending:
            self._say(f"{self.vid} skipped (click its button to come back).", "muted")
            self._open(pending[0])
        else:
            self._say("Nothing left to answer. Click “Finish & write document”.", "muted")
            self._render()

    def _pause(self) -> None:
        self._save()
        self._say("Draft saved. You can close the notebook; run the cells again to resume.", "ok")

    # --- citations -----------------------------------------------------------------------------

    def _add_citation(self) -> None:
        v = self.current_vignette()
        citations, status = ca.add_citation(self.cited, v.text, self.quote.value)
        if status == "added":
            self.entry["highlights"] = citations
            self.quote.value = ""
            self._say(f"Added citation {len(citations)}. Choose its direction.", "ok")
            self._save()
            self._render_citations()
            self._refresh_answer_text()
        elif status.startswith("duplicate:"):
            self.quote.value = ""
            self._say(f"That passage is already citation {status.split(':')[1]}.", "muted")
        elif status == "not_found":
            self._say("Those words aren't in the text. Paste an exact piece of it.", "warn")
        elif status == "too_long":
            self._say("That is too long for one citation. Paste a shorter passage.", "warn")

    def _set_direction(self, chip: int, direction) -> None:
        if chip <= self._locked_count():
            return
        for citation in self.cited:
            if citation["chip"] == chip:
                citation["direction"] = direction
        self._save()

    def _toggle_show(self, chip: int) -> None:
        self.shown_chip = None if self.shown_chip == chip else chip
        self._render_vignette()
        self._render_citations()

    def _remove(self, chip: int) -> None:
        if chip <= self._locked_count():
            return self._say(f"Citation {chip} is locked.", "warn")
        if chip not in ca.uncited(self.cited, self.entry["answers"]) and self.confirm_remove != chip:
            self.confirm_remove = chip
            return self._say(
                f"Citation {chip} is cited in your answers. Click Remove again to take it out of "
                "them too. Later citations move up one number.",
                "warn",
            )
        self._cancel_confirmations()
        self.shown_chip = None
        self.entry["highlights"], self.entry["answers"] = ca.remove_citation(
            self.cited, self.entry["answers"], chip
        )
        self._say(f"Removed citation {chip}.", "muted")
        self._save()
        self._render()

    def _cite_box(self):
        """The answer box on screen that can take a citation, if there is one."""
        v, step = self.current_vignette(), self.steps[self.k]
        if step == ca.REVIEW or (av.gate_locked(self.entry) and step in av.gate_ids(v)):
            return None
        kind = QUESTIONS[step].kind
        if kind == "open":
            return self.text
        if kind == "shift" and (self.entry.get("q4") or {}).get("shift"):
            return self.q4_why
        return None

    def _cite(self, chip: int) -> None:
        box = self._cite_box()
        if box is None:
            return
        # A text box doesn't tell where the cursor is, so the number goes at the end.
        before = box.value
        box.value = f"{before}{'' if not before or before[-1].isspace() else ' '}[{chip}] "

    # --- typing --------------------------------------------------------------------------------

    def _store_answer(self, step: str, shown: str) -> None:
        self.entry["answers"][step] = ca.to_tokens(shown, len(self.cited))
        self._cancel_confirmations()
        self._refresh_answer_text()
        if time.monotonic() - self.last_save > self.SAVE_EVERY:
            self._save()

    def _on_text(self, change) -> None:
        if not (self.loading or self.vid is None):
            self._store_answer(self.steps[self.k], change["new"])

    def _on_q4_why(self, change) -> None:
        if not (self.loading or self.vid is None):
            self._store_answer("Q4", change["new"])

    def _on_rating(self, change) -> None:
        if self.loading or self.vid is None or change["new"] is None:
            return
        self.entry["rating"] = None if change["new"] == "i" else change["new"]
        self.entry["insufficient"] = change["new"] == "i"
        self._save()

    def _on_q4(self, field: str, value) -> None:
        if self.loading or self.vid is None:
            return
        q4 = self.entry.setdefault("q4", {})
        q4[field] = None if value == "" else value
        if field in ("shift", "toward"):
            self._show_q4()
            self._render_citations()
        self._save()

    def _show_q4(self) -> None:
        """Lay out the Q4 widgets for the chosen shift: direction for +1/+2, why unless 0."""
        q4 = self.entry.get("q4") or {}
        shift = q4.get("shift")
        kids: list[W.Widget] = [
            W.HTML(f'<div class="av-q">{html.escape(Q4_HYPOTHESIS)}</div>'),
            self.q4_hyp,
            W.HTML(f'<div class="av-q">{html.escape(SHIFT_LEAD)}</div>'),
            self.q4_shift,
        ]
        if shift == 0:
            kids.append(W.HTML('<div class="av-muted">No change: <b>Next</b> continues.</div>'))
        if shift is not None and shift > 0:
            kids += [W.HTML(f'<div class="av-q">{html.escape(Q4_TOWARD)}</div>'), self.q4_toward]
            if q4.get("toward") == "other":
                kids.append(self.q4_other)
        if shift:
            kids += [
                W.HTML(f'<div class="av-q">{html.escape(Q4_WHY)}</div>'),
                self.q4_why,
                self.q4_helper,
            ]
        self.q4_box.children = kids

    def _answer_feedback(self, step: str, stored: str, minimum: int | None) -> str:
        """The counter, the answer with its chips in colour, and anything to look at."""
        words = ca.count_words(stored)
        counter = f"{words} of {minimum} different words" if minimum and words < minimum else (
            f"{words} different words"
        )
        parts = [f'<div class="av-muted">{counter}</div>']
        if ca.CHIP_RE.search(stored):
            parts.append(f'<div class="av-box">{answer_html(stored)}</div>')
        for number in ca.unknown_numbers(ca.to_display(stored), len(self.cited)):
            parts.append(f'<div class="av-warn">[{number}] is not one of your citations.</div>')
        if step == "Q1" and ca.q1_nudge(stored):
            parts.append(
                '<div class="av-warn">Which fragments is this about? '
                "Use Cite to link them.</div>"
            )
        return "".join(parts)

    def _refresh_answer_text(self) -> None:
        step = self.steps[self.k]
        if step == ca.REVIEW or QUESTIONS[step].kind == "rating_1_5":
            return
        stored = self.entry["answers"].get(step, "")
        minimum = ca.QUESTIONS[step].min_words
        target = self.q4_helper if step == "Q4" else self.helper
        target.value = self._answer_feedback(step, stored, minimum)
        self.next_btn_label()

    # --- drawing -------------------------------------------------------------------------------

    def _render_vignette(self) -> None:
        v = self.current_vignette()
        shown = next((c for c in self.cited if c["chip"] == self.shown_chip), None)
        self.vignette.value = (
            f'<div class="av-muted">{v.id}: {v.framing.replace("_", " ")}, {v.word_count} words</div>'
            f'<div class="av-box">{highlight(v.text, shown)}</div>'
        )

    def _citation_row(self, citation: dict, can_cite: bool) -> W.HBox:
        chip = citation["chip"]
        locked = chip <= self._locked_count()
        label = W.HTML(
            f'<div class="av-quote">{chip_html(chip)} “{html.escape(citation["text"])}”</div>',
            layout=W.Layout(flex="1 1 auto", width="auto"),
        )
        small = W.Layout(width="70px", flex="0 0 auto")
        show = button("Hide" if self.shown_chip == chip else "Show", layout=small)
        show.on_click(lambda _: self._toggle_show(chip))
        kids = [label]
        if locked:
            shown = dict(DIRECTIONS)
            names = {value: name for name, value in shown.items()}
            kids.append(
                W.HTML(
                    f'<div class="av-muted">{names.get(citation["direction"], "")}, locked</div>',
                    layout=W.Layout(width="170px", flex="0 0 auto"),
                )
            )
        else:
            direction = W.Dropdown(
                options=DIRECTIONS,
                value=citation.get("direction"),
                layout=W.Layout(width="170px", flex="0 0 auto"),
            )
            direction.observe(lambda c: self._set_direction(chip, c["new"]), "value")
            kids.append(direction)
        kids.append(show)
        if can_cite:
            cite = button("Cite", layout=small)
            cite.on_click(lambda _: self._cite(chip))
            kids.append(cite)
        if not locked:
            remove = button("Remove", layout=W.Layout(width="80px", flex="0 0 auto"))
            remove.on_click(lambda _: self._remove(chip))
            kids.append(remove)
        return W.HBox(kids, layout=W.Layout(gap="6px", align_items="center"))

    def _render_citations(self) -> None:
        can_cite = self._cite_box() is not None
        rows = [self._citation_row(citation, can_cite) for citation in self.cited]
        if not rows:
            rows = [W.HTML('<div class="av-muted">No citations yet.</div>')]
        self.citations.children = [W.HTML('<div class="av-q">Citations</div>'), *rows]

    def next_btn_label(self) -> None:
        step = self.steps[self.k]
        self.next_btn.button_style = "primary"
        self.next_btn.icon = "check" if step == ca.REVIEW else "arrow-right"
        if step == ca.REVIEW:
            self.next_btn.description = "Accept & next vignette"
        elif self._is_lock_step():
            self.next_btn.description = "Lock in results"
            self.next_btn.icon = "lock"
        elif self.k == len(self.steps) - 2 or self.edit_from_review:
            self.next_btn.description = "Review"
        else:
            self.next_btn.description = "Next"

    def _render(self) -> None:
        v, step, e = self.current_vignette(), self.steps[self.k], self.entry
        done, skipped = self._done(), self.draft["skipped"]
        self.progress.value = len(done)
        self.counts.value = (
            f'<span class="av-muted">{len(done)} done, {len(skipped)} skipped, '
            f"{len(self._pending())} to go</span>"
        )
        jump = []
        for vid in self.draft["order"]:
            style = (
                "info" if vid == self.vid else "success" if vid in done
                else "warning" if vid in skipped else ""
            )  # fmt: skip
            target = button(vid, style=style, layout=W.Layout(width="64px"), tooltip=f"open {vid}")
            target.on_click(lambda _, vid=vid: self._open(vid))
            jump.append(target)
        self.jump.children = jump

        self._render_vignette()
        self.loading = True
        count = f"question {self.k + 1} of {len(self.steps) - 1}"
        if step == ca.REVIEW:
            self.qhead.value = '<div class="av-q">Review your answers</div>'
            self.review.children = self._review_widgets(v, e)
            self.area.children = [self.review]
        elif step == "Q6":
            self.qhead.value = (
                f'<div class="av-q">Q6: {html.escape(ca.question_text("Q6", v))}</div>'
                f'<div class="av-muted">{count}</div>'
            )
            self.rating.value = "i" if e.get("insufficient") else e.get("rating")
            self.area.children = [self.rating]
        elif step == "Q4":
            assert v.q4_whatif is not None  # Q4 is only asked when the vignette has a what-if
            whatif = v.q4_whatif.strip().rstrip("?.")
            head, tail = ca.question_text("Q4", v).split(whatif, 1)
            self.qhead.value = (
                f'<div class="av-q">Q4: {html.escape(head)}'
                f'<span class="av-whatif">{html.escape(whatif)}</span>'
                f"{html.escape(tail)}</div>"
                f'<div class="av-muted">{count}</div>'
            )
            q4 = e.get("q4") or {}
            self.q4_hyp.value = q4.get("hypothesis") or ""
            self.q4_shift.value = q4.get("shift")
            self.q4_toward.value = q4.get("toward")
            self.q4_other.value = q4.get("other_explanation") or ""
            self.q4_why.value = ca.to_display(e["answers"].get("Q4", ""))
            self._show_q4()
            self.area.children = [self.q4_box]
        else:
            locked = av.gate_locked(e) and step in av.gate_ids(v)
            self.qhead.value = (
                f'<div class="av-q">{step}: {html.escape(ca.question_text(step, v))}</div>'
                f'<div class="av-muted">{count}'
                + (", locked: you have seen the questions after this one" if locked else "")
                + "</div>"
            )
            self.text.value = ca.to_display(e["answers"].get(step, ""))
            self.text.disabled = locked
            self.area.children = [self.text, self.helper]
        self.loading = False
        self._render_citations()
        self._refresh_answer_text()

        self.back_btn.disabled = self.k == 0
        self.next_btn_label()
        self.finish_btn.description = "Finish & write document"
        self.finish_btn.button_style = "success"

    def _review_widgets(self, v, e) -> list:
        out = []
        locked = av.gate_ids(v) if av.gate_locked(e) else []
        for i, step in enumerate(self.steps[:-1]):
            question = html.escape(ca.question_text(step, v))
            if step == "Q4":
                q4 = av.clean_q4(e.get("q4"))
                body = html.escape(av.q4_summary(q4))
                if q4 and q4["hypothesis"]:
                    body = f"hypothesis: {html.escape(q4['hypothesis'])}\n{body}"
                if q4 and q4["shift"] and e["answers"].get("Q4", "").strip():
                    body += f"\nwhy: {answer_html(e['answers']['Q4'].strip())}"
            elif step == "Q6":
                body = (
                    "insufficient information"
                    if e.get("insufficient")
                    else e.get("rating", "(not rated)")
                )
            else:
                body = (
                    answer_html(e["answers"].get(step, "").strip())
                    or '<span class="av-muted">(blank)</span>'
                )
            head = f'<div class="av-q">{step}: {question}</div><div class="av-box">{body}</div>'
            if step in locked:
                out.append(W.HTML(head + '<div class="av-muted">locked</div>'))
                continue
            edit = button(f"Edit {step}", "pencil", layout=W.Layout(width="110px"))
            edit.on_click(lambda _, i=i: self._edit(i))
            out += [W.HTML(head), edit]
        warnings = "".join(
            f'<div class="av-warn">{html.escape(w)}</div>' for w in ca.submit_warnings(e)
        )
        if warnings:
            out.append(W.HTML(warnings))
        return out

    def _edit(self, k: int) -> None:
        self.edit_from_review = True
        self._go(k)

    # --- finishing -----------------------------------------------------------------------------

    def _finish(self) -> None:
        done = self._done()
        if not done:
            return self._say("Nothing is finished yet: accept at least one vignette.", "warn")
        if not self.confirm:
            left = len(self.draft["order"]) - len(done)
            self.confirm = True
            self.finish_btn.description = f"Confirm: write {len(done)} vignette(s)"
            self.finish_btn.button_style = "danger"
            return self._say(
                f"{len(done)} finished vignette(s) go into the document."
                + (
                    f" {left} skipped or unfinished one(s) are left out and their drafts deleted."
                    if left
                    else ""
                )
                + " Click again to confirm, or anything else to cancel.",
                "warn",
            )
        try:
            doc = ca.finish_draft(self.gold, self.draft, self.vignettes, self.source_name)
        except Exception as e:  # show it, keep the draft
            self.confirm = False
            self._render()
            return self._say(f"Could not write the document: {html.escape(str(e))}", "bad")
        for w in (self.back_btn, self.next_btn, self.skip_btn, self.save_btn, self.finish_btn):
            w.disabled = True
        self.add_btn.disabled = True
        self.jump.children = []
        self.citations.children = []
        self.area.children = []
        self.qhead.value = ""
        self._say(
            f"Wrote <b>{doc.session_id}</b> with {len(doc.responses)} vignette(s). Thank you! "
            "Next: <code>python scripts/elicitation/make_gold_claims.py --source chips</code>",
            "ok",
        )


dashboard = AnswerDashboard()
display(dashboard.root)

After finishing, an agent writes the reference claims from the answers:
`python scripts/elicitation/make_gold_claims.py --source chips` (see `notes/fixtures.md`).